# SOCIOWEAVE Core 18 — Colab Experiment Runtime
Run real E1–E9 campaigns, aggregate paired observations and generate plots. Synthetic smoke-test data are isolated and never count as evidence.

In [ ]:
import sys, platform, subprocess, pathlib, os
print(sys.version, platform.platform())
try: print(subprocess.check_output(['nvidia-smi'],text=True))
except Exception as e: print('No visible NVIDIA GPU:',e)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT=pathlib.Path('/content/drive/MyDrive/SOCIOWEAVE')
DATA_DIR=ROOT/'datasets'; RUN_DIR=ROOT/'runs'/'core18'
DATA_DIR.mkdir(parents=True,exist_ok=True); RUN_DIR.mkdir(parents=True,exist_ok=True)


In [ ]:
GIT_COMMIT='ac3c95e12059d711e33974a67942bdfc79822d6f'  # reviewed exact SHA; update only after review
REPO='https://github.com/msellamiTN/socioweave-studio.git'
subprocess.check_call(['git','clone',REPO,'/content/socioweave-studio'])
subprocess.check_call(['git','-C','/content/socioweave-studio','checkout',GIT_COMMIT])
subprocess.check_call([sys.executable,'-m','pip','install','-e','/content/socioweave-studio/engine'])
sys.path.insert(0,'/content/socioweave-studio/engine')
from sociodec.datasets import load_jsonl_dataset
from sociodec.experiments import ExperimentConfig
from sociodec.evaluation import load_observations_jsonl,aggregate_observations,paired_comparison,correct_comparison_family,write_evaluation_outputs


## Dataset
Place a licensed JSONL snapshot in Drive. Every row needs example_id and prompt_text; optional fields: split and metadata. Preserve exact source version and split.

In [ ]:
DATASET_NAME='BREAK'  # BREAK / StrategyQA / TaskBench / TravelPlanner / smart-administration
DATASET_VERSION='record-source-release-and-split'
DATASET_PATH=DATA_DIR/f'{DATASET_NAME}.jsonl'
if not DATASET_PATH.exists(): raise FileNotFoundError(DATASET_PATH)
examples=load_jsonl_dataset(DATASET_PATH,dataset=DATASET_NAME,source_version=DATASET_VERSION)
print('Loaded prompts:',len(examples))


## Paper-aligned experiment configuration
Reference settings from the paper: five repeats, n=3 agents, k=3 reviewers, T≤2; temperatures 0.7/0.2/0; max tokens 1024/512; fixed executor. These are defaults, not a claim about a run.

In [ ]:
config=ExperimentConfig(experiment_id='E1',dataset=DATASET_NAME,
 model_families=('MODEL_FAMILY_A','MODEL_FAMILY_B','MODEL_FAMILY_C'),
 executor_model='FIXED_EXECUTOR_MODEL_VERSION',
 strategy_ids=('S1','S12','S2','S3','S4','S9','S5','S6','S8','S13','S7','S11','S10'),
 n_agents=3,k_experts=3,max_rounds=2,repeats=5,base_seed=17,
 temperature_draft=.7,temperature_critique=.2,temperature_decision=0.,
 max_tokens_decomposition=1024,max_tokens_other=512)
print('Config hash:',config.config_hash)


## Provider-backed campaign adapter — required
The current repository does not yet ship a fully validated provider-backed campaign adapter. Implement and test one that invokes the SocioDec orchestrator, executes outputs with the same fixed executor, and writes one observation row per dataset/example/strategy/repeat. Keep secrets in Colab Secrets/environment variables. Do not print credentials.

In [ ]:
def run_provider_backed_campaign(config,examples,output_path):
    raise NotImplementedError('Connect a validated provider-backed strategy and fixed-executor adapter before real E1–E9 runs.')
# Uncomment only after adapter integration and provider contract tests pass.
# run_provider_backed_campaign(config,examples,RUN_DIR/'observations.jsonl')


## Load real observations and evaluate
Each JSONL row: dataset, example_id, strategy_id, repeat_index, status, metrics; optional calls, depth, tokens, latency_seconds, failure_type. Failed attempts remain in the data.

In [ ]:
OBS_PATH=RUN_DIR/'observations.jsonl'
if not OBS_PATH.exists(): raise FileNotFoundError('Run validated provider adapter first; no synthetic fallback is used.')
observations=load_observations_jsonl(OBS_PATH)
summary=aggregate_observations(observations)
summary[:5]


In [ ]:
BASELINE_ID='single-agent-baseline'; METRIC='utility'
comparisons=[]
for strategy in config.strategy_ids:
    if strategy!=BASELINE_ID:
        comparisons.append(paired_comparison(observations,dataset=DATASET_NAME,strategy_id=strategy,
            baseline_id=BASELINE_ID,metric=METRIC,seed=config.base_seed))
comparisons=correct_comparison_family(comparisons)
paths=write_evaluation_outputs(observations,RUN_DIR/'analysis',comparisons)
paths


## Plots from real observations
Matplotlib only. Mean ± 1.96 standard errors are descriptive intervals, not paired inferential intervals. The paired bootstrap intervals and Holm-adjusted p-values are in the JSON comparison export.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plot_rows=[]
for r in observations:
    if r.status=='success':
        for metric,value in r.metrics.items():
            plot_rows.append({'strategy_id':r.strategy_id,'example_id':r.example_id,'metric':metric,'value':value,
                              'calls':r.calls,'tokens':r.tokens,'latency_seconds':r.latency_seconds})
df=pd.DataFrame(plot_rows)
if df.empty: raise ValueError('No successful observations to plot')
for metric in sorted(df.metric.unique()):
    g=df[df.metric==metric].groupby('strategy_id').value.agg(['mean','std','count']).reset_index()
    g['se']=g['std'].fillna(0)/(g['count']**.5)
    fig,ax=plt.subplots(figsize=(11,5)); ax.bar(g.strategy_id,g['mean'],yerr=1.96*g.se,capsize=3)
    ax.set_title(f'{metric}: mean ± 1.96 SE (descriptive)'); ax.set_ylabel(metric); ax.tick_params(axis='x',rotation=45)
    fig.tight_layout(); fig.savefig(RUN_DIR/f'{metric}_by_strategy.png',dpi=160); plt.show(); plt.close(fig)


In [ ]:
# Calls vs utility
if 'utility' in df.metric.unique():
    u=df[(df.metric=='utility') & df.calls.notna()]
    if not u.empty:
        fig,ax=plt.subplots(figsize=(8,5))
        for name,g in u.groupby('strategy_id'): ax.scatter(g.calls,g.value,label=name,alpha=.7)
        ax.set_xlabel('Model calls'); ax.set_ylabel('Utility'); ax.set_title('Cost–utility observations')
        ax.legend(bbox_to_anchor=(1.02,1),loc='upper left'); fig.tight_layout()
        fig.savefig(RUN_DIR/'calls_vs_utility.png',dpi=160); plt.show(); plt.close(fig)
# Failure rates
fail=pd.DataFrame([{'strategy_id':r.strategy_id,'failed':int(r.status=='failed')} for r in observations])
if not fail.empty:
    fig,ax=plt.subplots(figsize=(10,4)); fail.groupby('strategy_id').failed.mean().plot(kind='bar',ax=ax)
    ax.set_ylabel('Failure rate'); ax.set_title('Trial failure rate by strategy'); fig.tight_layout()
    fig.savefig(RUN_DIR/'failure_rates.png',dpi=160); plt.show(); plt.close(fig)


## Synthetic plotting smoke test — NOT EXPERIMENTAL EVIDENCE
This isolated cell verifies that plotting works before a campaign. Never merge these values into observations or cite them.

In [ ]:
demo=pd.DataFrame({'strategy':['S1','S12','S2','S3'],'utility':[.62,.71,.66,.68]})
fig,ax=plt.subplots(figsize=(7,4)); ax.bar(demo.strategy,demo.utility); ax.set_ylim(0,1)
ax.set_title('SYNTHETIC DEMO ONLY — NOT EXPERIMENTAL RESULTS'); plt.show(); plt.close(fig)


## Scientific completion checklist
- Exact code SHA, environment, dependency versions, dataset source/version/split and licenses recorded.
- Matcher validated; gold annotations and annotator agreement available where required.
- Fixed executor and same-budget baselines verified.
- Failures and missing metrics retained.
- Prompt-paired tests and Holm family defined before interpretation.
- E9 order, seed and paraphrase robustness evaluated.
- Figures regenerated from real exported observations.

No strategy is declared superior from a plot or unadjusted p-value alone.